# 14 — Denoising autoencoders and latent representations

**Level:** Advanced · **Study time:** 75–90 minutes (including experiments)

**Prerequisites:** Notebooks 08–09.

## What you will be able to do
- Train without class labels using reconstruction targets.
- Encode an image into a small latent vector.
- Compare denoising with the unprocessed noisy input.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 13](13_attention_and_transformers.ipynb) · [Next: 15](15_checkpoints_and_inference.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Corrupt inputs while keeping clean targets

An autoencoder compresses and reconstructs data. A denoising autoencoder receives corrupted images but targets the clean originals. Generate fresh training noise so the model cannot memorize one fixed corruption. Validation noise is fixed for fair comparisons.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits = load_digits()
X = torch.tensor(digits.data, dtype=torch.float32) / 16
tr, va = train_test_split(np.arange(len(X)), test_size=0.25, random_state=SEED, stratify=digits.target)
loader = DataLoader(TensorDataset(X[tr]), batch_size=64, shuffle=True)
noise_scale = 0.35
noisy_val = (X[va] + noise_scale * torch.randn_like(X[va])).clamp(0, 1)


## Step 3 — Build a bottleneck

The encoder reduces 64 pixel values to 12 latent values; the decoder reconstructs 64 pixels. A sigmoid output respects the scaled pixel range. MSE is a simple reconstruction objective; it may favor smooth averages.


In [ ]:
class Denoiser(nn.Module):
    def __init__(self, latent_dim=12):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 48), nn.ReLU(), nn.Linear(48, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 48), nn.ReLU(), nn.Linear(48, 64), nn.Sigmoid())
    def forward(self, x):
        return self.decoder(self.encoder(x))
model = Denoiser()
assert model.encoder(X[:4]).shape == (4, 12)


## Step 4 — Train and inspect reconstructions

Pixel error is useful but incomplete. Inspect reconstructions for lost strokes and changed identity. We do not use the digit class labels in optimization.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
for epoch in range(30):
    model.train()
    for (clean_batch,) in loader:
        noisy_batch = (clean_batch + noise_scale * torch.randn_like(clean_batch)).clamp(0, 1)
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(noisy_batch), clean_batch)
        loss.backward(); optimizer.step()
model.eval()
with torch.no_grad():
    restored = model(noisy_val)
    baseline_mse = nn.functional.mse_loss(noisy_val, X[va]).item()
    denoised_mse = nn.functional.mse_loss(restored, X[va]).item()
print({"noisy_input_MSE": baseline_mse, "denoised_MSE": denoised_mse})
assert denoised_mse < baseline_mse
fig, axes = plt.subplots(3, 6, figsize=(10, 5))
for row, (data, name) in enumerate([(X[va], "clean"), (noisy_val, "noisy"), (restored, "restored")]):
    for col in range(6):
        axes[row, col].imshow(data[col].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(name)
plt.tight_layout(); plt.show()


## Your turn — practice before opening the solutions

1. Compute one reconstruction MSE per validation image and find the worst example.
2. Interpolate between the latent vectors of two images, then decode the path.
3. Measure reconstruction error for pure random noise. Why is that not enough to validate an anomaly detector?

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
errors = (restored - X[va]).square().mean(1)
print("Worst validation index:", errors.argmax().item(), "MSE:", errors.max().item())
with torch.no_grad():
    z = model.encoder(X[va[:2]])
    alpha = torch.linspace(0, 1, 6).unsqueeze(1)
    interpolated = model.decoder((1 - alpha) * z[0] + alpha * z[1])
    noise = torch.rand(100, 64)
    print("Random-noise reconstruction MSE:", (model(noise) - noise).square().mean().item())
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, pixels in zip(axes, interpolated):
    ax.imshow(pixels.reshape(8, 8), cmap="gray"); ax.axis("off")
plt.show()
# Real anomalies can reconstruct well; validate thresholds on labeled, representative examples.


## Before moving on

An ordinary autoencoder is not automatically a calibrated anomaly detector or a probabilistic generator. Latent interpolation is a useful probe, but random latent sampling need not produce meaningful images.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Unsupervised learning with autoencoders (PyTorch examples)](https://github.com/pytorch/examples/tree/main/vae)


[Course guide](../README.md) · [Previous: 13](13_attention_and_transformers.ipynb) · [Next: 15](15_checkpoints_and_inference.ipynb)
